# Ordered Logistic Regression Results for Knowledge Adoption in Rangeland Management: Exploration with `mlcroissant`
This notebook provides a step-by-step example for loading and exploring the FAIR² dataset with the `mlcroissant` library using the Croissant schema URL.

### Dataset Source
The dataset source is specified via Croissant schema URL and includes record sets, fields, and columns uniquely referenced by their `@id`.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. Make sure to use the dataset Croissant schema URL.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and columns, displaying their names and `@id` fields as specified in the Croissant schema.

In [ ]:
# List all record sets, their fields and columns, by their @id
print('Available record sets in the dataset:')
for record_set in dataset.record_sets:
    print(f"- RecordSet name: {record_set.name} | @id: {record_set.id}")
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - {field.name} | @id: {field.id}")
        if hasattr(field, 'columns') and field.columns is not None:
            print("      Columns:")
            for column in field.columns:
                print(f"        - {column.name} | @id: {column.id}")
    print()

## 3. Data Extraction
Extract data from each record set using its `@id` and load into pandas DataFrames. 

**Note**: Update the `record_sets_to_load` variable below to include only record sets intended for data analysis (e.g., those that have data rows). If you do not know the available record sets, refer to the printed overview above.

In [ ]:
# Select record sets to extract by their @id
# Update the list below based on overview output. Example assumes a record set with @id 'cr:OrderedLogitRegressionResults'

record_sets_to_load = []
# Automatically add available record sets (edit if needed for more specific sets):
record_sets_to_load = [record_set.id for record_set in dataset.record_sets]

dataframes = {}
for record_set_id in record_sets_to_load:
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for record set @id: {record_set_id} -> shape: {dataframes[record_set_id].shape}")

# If at least one DataFrame loaded, print columns of the first
if dataframes:
    example_record_set_id = next(iter(dataframes.keys()))
    print(f"Columns for record set '{example_record_set_id}':")
    print(dataframes[example_record_set_id].columns.tolist())
    dataframes[example_record_set_id].head()
else:
    print("No records extracted. Check overview above to update record set IDs.")

## 4. Exploratory Data Analysis (EDA)
Now, process the extracted DataFrame(s). Example includes filtering, normalization, and grouping. Reference field names only by their `@id`.

> *To try your own analyses, identify a numeric column's `@id` and a grouping field's `@id` from above.*

In [ ]:
# Pick record set and fields for EDA
if dataframes:
    record_set_id = example_record_set_id
    df = dataframes[record_set_id]

    # Attempt to auto-detect a numeric field (`@id`) for demo
    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break

    # Fallback if not detected
    if numeric_field_id is None:
        print("No numeric field detected. Please inspect the DataFrame and set `numeric_field_id` manually.")
    else:
        threshold = df[numeric_field_id].mean() if not np.isnan(df[numeric_field_id].mean()) else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records in '{record_set_id}' with '{numeric_field_id}' > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized '{numeric_field_id}' for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to group by another column (`@id`)
        group_field_id = None
        # Pick first non-numeric and non-object id column
        for col in df.columns:
            if col != numeric_field_id and df[col].dtype == object:
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"\nGrouped by '{group_field_id}':")
            print(grouped_df.head())
        else:
            print("\nNo suitable group field found for grouping.")
else:
    print("No data loaded for EDA. Please check data extraction above.")

## 5. Visualization
Visualize value distributions or relationships (e.g., via histogram or boxplot) for the columns chosen above (`@id` only).

In [ ]:
import matplotlib.pyplot as plt

if dataframes and 'numeric_field_id' in locals() and numeric_field_id:
    plt.figure(figsize=(7,4))
    df[numeric_field_id].hist(bins=20, color='skyblue', edgecolor='black')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.show()

    # If group_field_id detected
    if 'group_field_id' in locals() and group_field_id:
        plt.figure(figsize=(8,5))
        df.boxplot(column=numeric_field_id, by=group_field_id, grid=False)
        plt.title(f"Boxplot of '{numeric_field_id}' by '{group_field_id}'")
        plt.suptitle('')
        plt.ylabel(numeric_field_id)
        plt.xlabel(group_field_id)
        plt.show()

## 6. Conclusion
This notebook demonstrated loading, overview, and analysis of a FAIR² Croissant dataset using the `mlcroissant` Python library. 

Key findings and possible next steps:
- Explored available record sets, fields, and used their `@id` for all references.
- Loaded data into pandas DataFrames and performed basic EDA.
- Visualized numeric field distributions and grouped statistics for further insights.

For full analysis, refer to the schema overview to select relevant `@id`s for deeper exploration or build custom visualizations using the variable DataFrame(s) above.